# Domain modelling with JAX pytrees

A model can receive a named domain object instead of a collection of opaque feature tensors. This example keeps a policyholder and a contract together as one `portfolio` solve input, then passes structured derived fields through the model and cashflow graph.

Run this notebook from a checkout after `pip install -e .`. It uses JAX, NumPy, and `jact` only.


In [ ]:
from dataclasses import dataclass, replace

import jax
import jax.numpy as jnp
import numpy as np
import jact


## 1. Define the domain objects

JAX knows how to transform a pytree through JIT and differentiation. A plain dataclass needs registration; `tree_flatten` declares its dynamic leaves and `tree_unflatten` rebuilds the object. Here every field is a numeric leaf, including the contract's shared scalar rates.


In [ ]:
@jax.tree_util.register_pytree_node_class
@dataclass(frozen=True)
class Policyholder:
    entry_age: jax.Array
    smoker: jax.Array

    def tree_flatten(self):
        return (self.entry_age, self.smoker), None

    @classmethod
    def tree_unflatten(cls, aux, children):
        return cls(*children)


@jax.tree_util.register_pytree_node_class
@dataclass(frozen=True)
class Contract:
    annual_premium: jax.Array
    disability_benefit: jax.Array
    indexation: jax.Array
    discount_rate: jax.Array

    def tree_flatten(self):
        return (
            self.annual_premium,
            self.disability_benefit,
            self.indexation,
            self.discount_rate,
        ), None

    @classmethod
    def tree_unflatten(cls, aux, children):
        return cls(*children)


portfolio = {
    "person": Policyholder(
        entry_age=jnp.array([35.0, 48.0, 61.0]),
        smoker=jnp.array([0.0, 1.0, 0.0]),
    ),
    "contract": Contract(
        annual_premium=jnp.array([700.0, 850.0, 1000.0]),
        disability_benefit=jnp.array([9000.0, 12000.0, 15000.0]),
        indexation=jnp.array(0.02),
        discount_rate=jnp.array(0.03),
    ),
}

for path, leaf in jax.tree_util.tree_flatten_with_path(portfolio)[0]:
    print(f"{jax.tree_util.keystr(path)}: {leaf.shape}")


Every nonscalar **solve-input** leaf uses axis 0 for individuals. All of those leaves must have the same leading size; scalar leaves such as `indexation` and `discount_rate` are shared. The domain structure survives the solve boundary, so model functions can use names and attributes rather than column positions.

## 2. Derive structured model features

The `profile` field depends on time and returns a dictionary. The `hazards` field consumes that dictionary and the duration grid, then returns another dictionary. The solver evaluates each field in the relevant context and passes the complete result to transition functions.


In [ ]:
state_space = jact.StateSpace(
    states=["active", "disabled", "dead"],
    transitions=[
        ("active", "disabled"),
        ("active", "dead"),
        ("disabled", "dead"),
    ],
)


def profile(t, portfolio):
    person = portfolio["person"]
    contract = portfolio["contract"]
    return {
        "attained_age": person.entry_age + t,
        "smoker_factor": 1.0 + 0.5 * person.smoker,
        "benefit": contract.disability_benefit * (1.0 + contract.indexation * t),
    }


def hazards(d, profile):
    base = (
        0.006
        * jnp.exp(0.035 * (profile["attained_age"][:, None] - 50.0))
        * profile["smoker_factor"][:, None]
    )
    return {
        "onset": base * (1.0 + 0.04 * d),
        "death": 0.35 * base * (1.0 + 0.02 * d),
    }


model = state_space.build(
    transitions={
        ("active", "disabled"): lambda t, d, **kw: kw["hazards"]["onset"],
        ("active", "dead"): lambda t, d, **kw: kw["hazards"]["death"],
        ("disabled", "dead"): lambda t, d, **kw: 2.0 * kw["hazards"]["death"],
    },
    derived={"profile": profile, "hazards": hazards},
)


## 3. Use the same objects in cashflows

The cashflow declaration adds a `payments` dictionary derived from the model's `profile` and the original `portfolio`. The solve adds a discount field for a terminal present-value view. Each scope owns its definitions, but all fields participate in one dependency graph.


In [ ]:
cashflows = state_space.cashflows(
    {
        "premium": jact.cashflows.StateRate(
            {"active": lambda t, d, **kw: -kw["payments"]["premium"][:, None] + 0 * d}
        ),
        "disability": jact.cashflows.StateRate(
            {"disabled": lambda t, d, **kw: kw["payments"]["benefit"][:, None] + 0 * d}
        ),
    },
    derived={
        "payments": lambda profile, portfolio: {
            "premium": portfolio["contract"].annual_premium,
            "benefit": profile["benefit"],
        }
    },
)

views = {
    "raw": jact.cashflows.Raw(),
    "present_value": jact.cashflows.Total(
        weight=lambda t, **kw: kw["discount_factor"], terminal=True
    ),
}
solve_fields = {
    "discount_factor": lambda t, portfolio: jnp.exp(
        -portfolio["contract"].discount_rate * t
    )
}

result = model.solve(
    initial="active",
    horizon=2,
    steps_per_unit=4,
    cashflows=cashflows,
    cashflow_views=views,
    derived=solve_fields,
    portfolio=portfolio,
)

print("states:", result.states)
print("probability shape:", result.probability.shape)
print("present values:", np.round(np.asarray(result.cashflows["present_value"]), 2))
assert result.probability.shape == (9, 3, 3)
assert result.cashflows["present_value"].shape == (3,)


## 4. Keep the domain structure under JIT and differentiation

The registered classes remain pytrees when JAX traces the solve. Here the contract's scalar indexation rate is a differentiable leaf; changing it leaves the portfolio structure unchanged.


In [ ]:
@jax.jit
def present_value(p):
    return model.solve(
        initial="active",
        horizon=2,
        steps_per_unit=4,
        cashflows=cashflows,
        cashflow_views=views,
        derived=solve_fields,
        portfolio=p,
    ).cashflows["present_value"]


compiled_value = present_value(portfolio)
np.testing.assert_allclose(compiled_value, result.cashflows["present_value"])


def total_value_at_indexation(rate):
    changed_contract = replace(portfolio["contract"], indexation=rate)
    changed_portfolio = {**portfolio, "contract": changed_contract}
    return jnp.sum(present_value(changed_portfolio))


print("d(total present value)/d(indexation):", jax.grad(total_value_at_indexation)(jnp.array(0.02)))


## 5. Catch a malformed cohort at the boundary

A mismatch is reported with the input and leaf path. With the positional registration above, a class leaf appears as `<flat index 0>` rather than the Python attribute name. A solve-input vector always establishes or follows the batch size; keep shared nonscalar parameters in a closure or another explicitly modelled static component.


In [ ]:
bad_person = replace(portfolio["person"], entry_age=jnp.array([35.0, 48.0]))
bad_portfolio = {**portfolio, "person": bad_person}
try:
    model.solve(
        initial="active",
        horizon=2,
        steps_per_unit=4,
        portfolio=bad_portfolio,
    )
except ValueError as exc:
    print(exc)
    assert "portfolio" in str(exc) and "person" in str(exc) and "flat index 0" in str(exc)
else:
    raise AssertionError("Expected a batch-size error")


The numerical outputs are still arrays. Pytrees organize the **inputs and derived data** that produce those arrays; they do not change the intensity and payment output-shape contracts. Keep the tree structure and leaf shapes stable across repeated JIT calls to reuse compiled work.
